In [0]:
# Run configuration notebook to set up paths and parameters
%run /Workspace/Users/sawantr2305@gmail.com/HealthCare_DB_Pipeline/src/config/config.ipynb

In [0]:
# Define bronze data path and print it for verification
bronze_path = f"{RAW_PATH}/bronze/{bronze_catalog}"
print(bronze_path)

In [0]:
# Load CSV data from bronze path into Spark DataFrame
new_df = spark.read.format('csv').options(header=True, inferSchema=True).load(bronze_path)

In [0]:
# Display first 50 rows of the DataFrame for inspection
display(new_df.limit(50))

In [0]:
# Import Spark SQL functions for data processing
from pyspark.sql import functions as F

In [0]:
# Replace specific string values in 'Age' column for normalization
word_map = {"forty": "40", "nan": "46"}
new_df = new_df.replace(word_map, subset=["Age"])

In [0]:
# Replace 'NaN' values in 'Blood Pressure' column with default value
new_df = new_df.replace({'NaN' : '120/80'}, subset=["Blood Pressure"])

In [0]:
# Clean 'Cholesterol' column, replace missing values, and cast to integer
new_df = new_df.withColumn(
    'Cholesterol',
    F.when(F.col("Cholesterol").isin(['NaN', 'nan']), '160')
    .otherwise(F.col("Cholesterol"))
    .cast('float')
    .cast('int')
)

In [0]:
# Replace missing values in 'Email' column and fill other nulls with default email
new_df = new_df.replace({'nan' : 'patient@example.com'}, subset=["Email"]).fillna('name@hospital')

In [0]:
# Replace missing and invalid values in 'Phone Number' column with default phone numbers
new_df = (
    new_df.replace({'nan': '123-456-7890'}, subset=["Phone Number"])
    .replace({' ': '098-765-4321'}, subset=["Phone Number"])
)

In [0]:
# Parse 'Visit Date' column using multiple date formats and convert to date type
new_df = new_df.withColumn(
    "Visit Date",
    F.coalesce(
        F.try_to_date("Visit Date", "MMMM d, yyyy"),
        F.try_to_date("Visit Date", "yyyy/MM/dd"),
        F.try_to_date("Visit Date", "MM-dd-yyyy"),
        F.try_to_date("Visit Date", "MM/dd/yyyy"),
        F.try_to_date("Visit Date", "yyyy.MM.dd")
    )
)

In [0]:
# Save the cleaned DataFrame to the silver layer in Parquet format
silver_path = f"{RAW_PATH}/silver/{silver_catalog}"
new_df.write.mode("overwrite").parquet(silver_path)